# Landing → Bronze

Neste notebook, vamos receber os cinco arquivos de filmes e consultar as cotações do dólar no Banco Central. A Bronze mantém os dados de origem disponíveis antes dos tratamentos de negócio da Silver. Conferiremos as entradas, investigaremos a interpretação dos CSVs, prepararemos as tabelas e realizaremos a gravação em Delta com controle de reexecução. 

As tabelas auxiliares preservam o conteúdo de todas as linhas de dados lidas, enquanto as principais recebem os registros interpretáveis pela política adotada. A cobertura e as limitações dessa escolha serão apresentadas ao longo do notebook.

## Configuração e parâmetros da execução

Vamos centralizar os caminhos e os parâmetros usados na carga. Os widgets data_inicio e data_fim definem o período das cotações no formato MM-DD-AAAA e não filtram os filmes. Quando ambos ficam vazios, a consulta considera os últimos sete dias corridos, incluindo o dia da execução. 

O parâmetro id_lote identifica a carga nos metadados e recebe a data atual quando não é informado; ele não é necessariamente um identificador único de execução. O controle contra repetição dos dados independe desse valor. Utilizaremos America/Recife para determinar o dia da execução e UTC para representar os horários de ingestão no Spark.

In [0]:
import csv
import io
import hashlib
import json
import re

from datetime import datetime, timedelta
from decimal import Decimal
from zoneinfo import ZoneInfo

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    ArrayType,
    DecimalType,
)


catalogo = "workspace"
schema_bronze = "bronze"
caminho_inputs = f"/Volumes/{catalogo}/landing/inputs"

# Padronizamos a exibição dos horários de ingestão.
spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("data_inicio", "", "Data inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data final (MM-DD-AAAA)")
dbutils.widgets.text("id_lote", "", "Identificador da carga")

hoje = datetime.now(ZoneInfo("America/Recife")).date()

inicio_informado = dbutils.widgets.get("data_inicio").strip()
fim_informado = dbutils.widgets.get("data_fim").strip()
id_lote = dbutils.widgets.get("id_lote").strip() or hoje.isoformat()


def interpretar_data(texto):
    if not re.fullmatch(r"[0-9]{2}-[0-9]{2}-[0-9]{4}", texto):
        raise ValueError("Informe as datas no formato MM-DD-AAAA.")

    return datetime.strptime(texto, "%m-%d-%Y").date()


if bool(inicio_informado) != bool(fim_informado):
    raise ValueError("Preencha as duas datas ou deixe ambas vazias.")

data_inicio = (
    interpretar_data(inicio_informado)
    if inicio_informado
    else hoje - timedelta(days=6)
)

data_fim = (
    interpretar_data(fim_informado)
    if fim_informado
    else hoje
)

if data_inicio > data_fim:
    raise ValueError("A data inicial não pode ser posterior à final.")

if data_fim > hoje:
    raise ValueError("Não consultaremos cotações de datas futuras.")

print("Lote:", id_lote)
print("Período da API:", data_inicio, "até", data_fim)
print("Volume de entrada:", caminho_inputs)

Lote: 2026-09-21
Período da API: 2026-09-15 até 2026-09-21
Volume de entrada: /Volumes/workspace/landing/inputs


#### Resultado da configuração

Na execução salva nesta versão, o lote foi identificado como 2026-09-19 e o período consultado foi de 13/09/2026 a 19/09/2026. Os arquivos foram localizados a partir do volume /Volumes/workspace/landing/inputs. Esses valores descrevem a execução registrada e podem mudar quando os parâmetros ou a data de execução forem alterados.

## Arquivos de entrada e tabelas de destino

Vamos relacionar cada CSV à tabela Bronze correspondente e às colunas esperadas. O arquivo disponibilizado como movies_info_TMDB_IMDB.csv apresenta a ordem das siglas diferente da indicada no PDF, mas alimentará o destino exigido, tb_movies_info. Primeiro verificaremos a disponibilidade dos cinco arquivos; a conferência dos cabeçalhos e do conteúdo será realizada durante a preparação.

In [0]:
fontes = [
    {
        "arquivo": "movies_info_TMDB_IMDB.csv",
        "tabela": "tb_movies_info",
        "colunas": [
            "id",
            "tconst",
            "title",
            "original_title",
            "original_language",
            "release_date",
            "runtime",
            "status",
            "overview",
            "tagline",
        ],
    },
    {
        "arquivo": "movies_financials_IMDB_TMDB.csv",
        "tabela": "tb_movies_financials",
        "colunas": [
            "id",
            "budget",
            "revenue",
        ],
    },
    {
        "arquivo": "movies_metrics_IMDB_TMDB.csv",
        "tabela": "tb_movies_metrics",
        "colunas": [
            "id",
            "popularity",
            "vote_average",
            "vote_count",
            "averageRating",
            "numVotes",
        ],
    },
    {
        "arquivo": "credits_and_tags_IMDB_TMDB.csv",
        "tabela": "tb_credits_and_tags",
        "colunas": [
            "id",
            "genres",
            "production_companies",
            "production_countries",
            "spoken_languages",
            "keywords",
            "directors",
            "writers",
            "cast",
        ],
    },
    {
        "arquivo": "movies_reviews.csv",
        "tabela": "tb_movies_reviews",
        "colunas": [
            "id",
            "nome",
            "nota",
            "comentario",
        ],
    },
]

arquivos = dbutils.fs.ls(caminho_inputs)

nomes_disponiveis = {
    arquivo.name
    for arquivo in arquivos
}

faltantes = [
    fonte["arquivo"]
    for fonte in fontes
    if fonte["arquivo"] not in nomes_disponiveis
]

if faltantes:
    raise FileNotFoundError(
        f"Arquivos não encontrados no volume: {faltantes}"
    )

display(
    spark.createDataFrame(
        [
            (
                fonte["arquivo"],
                f"{catalogo}.{schema_bronze}.{fonte['tabela']}",
                len(fonte["colunas"]),
            )
            for fonte in fontes
        ],
        [
            "arquivo",
            "tabela_destino",
            "quantidade_colunas_esperada",
        ],
    )
)

arquivo,tabela_destino,quantidade_colunas_esperada
movies_info_TMDB_IMDB.csv,workspace.bronze.tb_movies_info,10
movies_financials_IMDB_TMDB.csv,workspace.bronze.tb_movies_financials,3
movies_metrics_IMDB_TMDB.csv,workspace.bronze.tb_movies_metrics,6
credits_and_tags_IMDB_TMDB.csv,workspace.bronze.tb_credits_and_tags,9
movies_reviews.csv,workspace.bronze.tb_movies_reviews,4


#### Resultado do mapeamento das fontes

Os cinco arquivos foram encontrados no volume e associados às tabelas de informações, dados financeiros, métricas, créditos e avaliações. As estruturas esperadas possuem, respectivamente, dez, três, seis, nove e quatro colunas. Essa verificação confirma a disponibilidade dos arquivos e o mapeamento dos destinos, mas ainda não garante que todos os registros possam ser interpretados corretamente.

## Exploração inicial dos valores ausentes

Vamos observar o arquivo financeiro sem converter suas colunas para números. Além de valores NULL, procuraremos textos vazios, espaços e os marcadores Unknown, Não Informado e N/A. A leitura exploratória ajuda a reconhecer como a ausência aparece na origem; ela não será utilizada como substituta da preparação e da conferência dos arquivos realizadas adiante.

In [0]:
df_financeiro_exploracao = (
    spark.read
    .option("header", True)
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(f"{caminho_inputs}/movies_financials_IMDB_TMDB.csv")
)

df_financeiro_exploracao.printSchema()
display(df_financeiro_exploracao.limit(5))

expressoes = []

for coluna in ["id", "budget", "revenue"]:
    valor = F.col(coluna)
    texto = F.lower(F.trim(valor))

    expressoes.extend([
        F.count(
            F.when(valor.isNull(), 1)
        ).alias(f"{coluna}_nulos"),

        F.count(
            F.when(F.trim(valor) == "", 1)
        ).alias(f"{coluna}_vazios"),

        F.count(
            F.when(
                texto.isin("unknown", "não informado", "n/a"),
                1,
            )
        ).alias(f"{coluna}_ausencia_textual"),
    ])

resultado_financeiro = (
    df_financeiro_exploracao
    .agg(
        F.count("*").alias("total_registros"),
        *expressoes,
    )
    .first()
)

print("Registros financeiros:", resultado_financeiro["total_registros"])

display(
    spark.createDataFrame(
        [
            (
                coluna,
                resultado_financeiro[f"{coluna}_nulos"],
                resultado_financeiro[f"{coluna}_vazios"],
                resultado_financeiro[f"{coluna}_ausencia_textual"],
            )
            for coluna in ["id", "budget", "revenue"]
        ],
        [
            "coluna",
            "nulos",
            "vazios_ou_espacos",
            "ausencia_textual",
        ],
    )
)

root
 |-- id: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)



id,budget,revenue
293660,58000000,Unknown
299536,300000000,2052415039
299534,356000000,2800000000
475557,55000000,1074458282
271110,250000000,Não Informado


Registros financeiros: 106165


coluna,nulos,vazios_ou_espacos,ausencia_textual
id,0,0,0
budget,0,0,6253
revenue,0,0,9404


#### Resultado da exploração financeira

Foram observados 106.165 registros financeiros. Nas colunas verificadas, não apareceram valores NULL ou vazios na leitura exploratória, mas encontramos 6.253 marcadores textuais de ausência em budget e 9.404 em revenue. 

Esses grupos podem envolver os mesmos registros e não devem ser somados como filmes distintos. Importante notar que contar apenas NULL não revela todas as ausências. A interpretação dos marcadores e a conversão dos valores financeiros serão realizadas na Silver.

## Investigação da interpretação dos CSVs

Encontramos registros cujas aspas e barras invertidas permitem interpretações diferentes. Vamos comparar dois leitores: ambos reconhecem aspas duplicadas, mas apenas o segundo considera a barra invertida como escape. 

A classificação será concordante quando os dois retornarem os mesmos campos na quantidade esperada; será ambígua quando pelo menos um apresentar a quantidade esperada sem haver concordância entre ambos; e será divergência estrutural quando nenhum apresentar essa quantidade. Mostraremos um exemplo para registrar o problema sem repetir levantamentos extensos.

In [0]:
def calcular_hash_arquivo(caminho):
    resumo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            resumo.update(bloco)

    return resumo.hexdigest()


def classificar_linha_csv(linha, quantidade_esperada):
    resultados = []
    observacoes = []

    for escape in [None, "\\"]:
        try:
            registros = list(csv.reader(
                io.StringIO(linha, newline=""),
                delimiter=",",
                quotechar='"',
                doublequote=True,
                escapechar=escape,
                strict=True,
            ))

            if len(registros) == 1:
                campos = registros[0]
                resultados.append(campos)
                observacoes.append(f"{len(campos)} campos")
            else:
                resultados.append(None)
                observacoes.append(
                    "Quantidade inesperada de registros"
                )

        except csv.Error as erro:
            resultados.append(None)
            observacoes.append(f"Erro: {erro}")

    sem_escape, com_escape = resultados

    estrutura_sem_escape = (
        sem_escape is not None
        and len(sem_escape) == quantidade_esperada
    )

    estrutura_com_escape = (
        com_escape is not None
        and len(com_escape) == quantidade_esperada
    )

    if (
        estrutura_sem_escape
        and estrutura_com_escape
        and sem_escape == com_escape
    ):
        situacao = "interpretacao_concordante"

    elif estrutura_sem_escape or estrutura_com_escape:
        situacao = "interpretacao_ambigua"

    else:
        situacao = "divergencia_estrutural"

    return (
        situacao,
        sem_escape,
        com_escape,
        observacoes[0],
        observacoes[1],
    )


estrutura_classificacao = StructType([
    StructField("situacao", StringType(), True),
    StructField(
        "campos_sem_escape",
        ArrayType(StringType()),
        True,
    ),
    StructField(
        "campos_com_escape",
        ArrayType(StringType()),
        True,
    ),
    StructField("observacao_sem_escape", StringType(), True),
    StructField("observacao_com_escape", StringType(), True),
])

classificar_csv_spark = F.udf(
    classificar_linha_csv,
    estrutura_classificacao,
    useArrow=False,
)

#### Resultado da investigação da interpretação dos CSVs

Foram definidas duas configurações de leitura: uma mantém as barras invertidas como conteúdo, enquanto a outra as considera caracteres de escape. A comparação distingue interpretações concordantes, ambíguas e divergências estruturais. Essa célula apenas prepara as funções utilizadas na investigação; ainda não apresenta um diagnóstico dos arquivos. Na próxima etapa, aplicaremos essas funções a um registro de métricas para observar as diferenças entre os campos retornados.

## Exemplo de divergência na interpretação

Vamos comparar as duas configurações de leitura em um registro de métricas. Exibiremos a linha original e os campos produzidos por cada leitor para observar como as aspas e barras invertidas afetam a interpretação.

In [0]:
fonte_metricas = next(
    fonte
    for fonte in fontes
    if fonte["tabela"] == "tb_movies_metrics"
)

caminho_metricas = (
    f"{caminho_inputs}/{fonte_metricas['arquivo']}"
)

with open(
    caminho_metricas,
    "r",
    encoding="utf-8-sig",
    newline="",
) as arquivo:

    next(arquivo)  # Cabeçalho.

    for numero_linha, conteudo in enumerate(arquivo, start=2):
        resultado = classificar_linha_csv(
            conteudo,
            len(fonte_metricas["colunas"]),
        )

        if resultado[0] != "interpretacao_concordante":
            print("Linha física:", numero_linha)
            print("Texto original:", repr(conteudo))
            print("Situação:", resultado[0])

            print("\nSem escape por barra:")
            print("Observação:", resultado[3])
            print("Campos:", resultado[1])

            print("\nCom escape por barra:")
            print("Observação:", resultado[4])
            print("Campos:", resultado[2])
            break

    else:
        print("Nenhuma divergência encontrada pelos critérios aplicados.")

Linha física: 36
Texto original: '324857,"causing others from across the Spider-Verse to be inadvertently transported to his dimension.\\"",8.404,,Phil Lord"," Rodney Rothman""",8.4\r\n'
Situação: divergencia_estrutural

Sem escape por barra:
Observação: 4 campos
Campos: ['324857', 'causing others from across the Spider-Verse to be inadvertently transported to his dimension.\\",8.404,,Phil Lord', ' Rodney Rothman"', '8.4']

Com escape por barra:
Observação: 7 campos
Campos: ['324857', 'causing others from across the Spider-Verse to be inadvertently transported to his dimension."', '8.404', '', 'Phil Lord"', ' Rodney Rothman"', '8.4']


#### Resultado da investigação e decisão de leitura

Na linha física 36 do arquivo de métricas, esperávamos seis campos, mas os leitores retornaram quatro e sete. Portanto, nenhuma das configurações resolveu esse registro. Para a preparação dos arquivos, adotaremos uma política explícita: priorizar a leitura com aspas duplicadas e barras invertidas preservadas como conteúdo; utilizar a alternativa com escape por barra somente quando a primeira não apresentar a quantidade esperada de campos e a alternativa apresentar. 

Registros incompatíveis com ambas permanecerão apenas nas auxiliares, que também preservarão as linhas dos registros aceitos e as duas interpretações. Essa escolha verifica a quantidade de campos, mas não garante que os valores estejam nas posições corretas.

## Preparação dos arquivos e cobertura da leitura

Vamos conferir os cabeçalhos e comparar a quantidade de linhas da origem com a leitura no Spark. Para estas fontes, a leitura por linha exige que cada registro comece com um identificador numérico seguido de vírgula; essa condição não torna a estratégia adequada a qualquer CSV. 

Priorizaremos a interpretação sem escape por barra e utilizaremos a alternativa apenas quando a primeira não apresentar a quantidade esperada de campos. As tabelas principais receberão os registros aceitos, mantendo seus campos como texto e suas repetições. As auxiliares preservarão todas as linhas de dados e as duas interpretações. Ao final, verificaremos se os registros aceitos e pendentes explicam integralmente a quantidade recebida.

In [0]:
cargas = []
controle_fontes = []

for fonte in fontes:
    caminho = f"{caminho_inputs}/{fonte['arquivo']}"
    assinatura = calcular_hash_arquivo(caminho)
    quantidade_esperada = len(fonte["colunas"])

    # Conferimos o cabeçalho e contamos as linhas de dados.
    with open(
        caminho,
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as arquivo:
        primeira_linha = next(arquivo, None)

        if primeira_linha is None:
            raise ValueError(f"Arquivo vazio: {fonte['arquivo']}")

        cabecalho_texto = primeira_linha.rstrip("\r\n")

        cabecalho = next(
            csv.reader([cabecalho_texto], strict=True)
        )

        if cabecalho != fonte["colunas"]:
            raise ValueError(
                f"Cabeçalho inesperado em {fonte['arquivo']}: "
                f"{cabecalho}"
            )

        total_origem = sum(1 for _ in arquivo)

    if total_origem == 0:
        raise ValueError(
            f"Arquivo sem registros: {fonte['arquivo']}"
        )

    # Preservamos o conteúdo de cada linha antes de interpretar os campos.
    df_texto = (
        spark.read.text(caminho)
        .withColumnRenamed("value", "linha_original")
    )

    quantidade_cabecalhos = (
        df_texto
        .filter(F.col("linha_original") == cabecalho_texto)
        .count()
    )

    if quantidade_cabecalhos != 1:
        raise ValueError(
            f"Cabeçalho não identificado de forma única: "
            f"{fonte['arquivo']}"
        )

    df_original = df_texto.filter(
        F.col("linha_original") != cabecalho_texto
    )

    verificacao = df_original.agg(
        F.count("*").alias("total"),
        F.count(
            F.when(
                ~F.col("linha_original").rlike(r"^[0-9]+,"),
                1,
            )
        ).alias("inicio_diferente"),
    ).first()

    if verificacao["total"] != total_origem:
        raise ValueError(
            f"Contagem diferente da origem: {fonte['arquivo']}"
        )

    if verificacao["inicio_diferente"] > 0:
        raise ValueError(
            f"{fonte['arquivo']} possui linhas que não começam "
            "com ID e vírgula. A leitura por linha precisa ser "
            "revista para esta fonte."
        )

    # Guardamos as duas interpretações para permitir a conferência.
    df_classificado = (
        df_original
        .withColumn(
            "classificacao",
            classificar_csv_spark(
                F.col("linha_original"),
                F.lit(quantidade_esperada),
            ),
        )
        .select(
            "linha_original",
            "classificacao.*",
        )
    )

    sem_escape_valido = (
        F.col("campos_sem_escape").isNotNull()
        & (F.size("campos_sem_escape") == quantidade_esperada)
    )

    com_escape_valido = (
        F.col("campos_com_escape").isNotNull()
        & (F.size("campos_com_escape") == quantidade_esperada)
    )

    # Priorizamos a configuração que preserva as barras invertidas.
    # A alternativa é usada somente quando a primeira não apresenta
    # a quantidade esperada de campos.
    df_interpretado = df_classificado.withColumn(
        "campos_escolhidos",
        F.when(
            sem_escape_valido,
            F.col("campos_sem_escape"),
        ).when(
            com_escape_valido,
            F.col("campos_com_escape"),
        ),
    )

    # A seleção verifica a estrutura, não a validade dos valores.
    # Todas as colunas continuam como texto.
    df_principal = (
        df_interpretado
        .filter(F.col("campos_escolhidos").isNotNull())
        .select(
            *[
                F.col("campos_escolhidos")
                .getItem(posicao)
                .alias(coluna)
                for posicao, coluna in enumerate(fonte["colunas"])
            ]
        )
    )

    resumo_leitura = df_interpretado.agg(
        F.count("*").alias("total"),
        F.count("campos_escolhidos").alias("principal"),
        F.count(
            F.when(
                F.col("campos_escolhidos").isNull(),
                1,
            )
        ).alias("pendentes"),
    ).first()

    total_principal = resumo_leitura["principal"]
    total_pendente = resumo_leitura["pendentes"]

    if (
        resumo_leitura["total"] != total_origem
        or total_principal + total_pendente != total_origem
    ):
        raise ValueError(
            f"As contagens da leitura não fecham: {fonte['arquivo']}"
        )

    # A auxiliar mantém todas as linhas e as duas interpretações.
    # A principal recebe os registros interpretáveis pela política atual.
    destinos_fonte = [
        (
            f"{fonte['tabela']}_raw",
            df_classificado,
            total_origem,
            "concordancia_dois_leitores_v1",
        ),
        (
            fonte["tabela"],
            df_principal,
            total_principal,
            "leitura_prioritaria_sem_escape_v2",
        ),
    ]

    for nome_tabela, df, quantidade, politica in destinos_fonte:
        cargas.append({
            "tabela": nome_tabela,
            "df": df,
            "quantidade": quantidade,
            "assinatura": assinatura,
            "origem": caminho,
            "politica": politica,
            "total_origem": total_origem,
            "total_pendente": total_pendente,
        })

    controle_fontes.append((
        fonte["tabela"],
        total_origem,
        total_principal,
        total_pendente,
    ))

df_cobertura = (
    spark.createDataFrame(
        controle_fontes,
        [
            "tabela",
            "registros_origem",
            "registros_principal",
            "registros_pendentes",
        ],
    )
    .withColumn(
        "percentual_estruturado",
        F.round(
            F.col("registros_principal")
            / F.col("registros_origem")
            * 100,
            2,
        ),
    )
)

display(df_cobertura.orderBy("tabela"))

total_pendencias = sum(
    linha[3] for linha in controle_fontes
)

print(
    "Registros preservados somente nas auxiliares:",
    total_pendencias,
)

tabela,registros_origem,registros_principal,registros_pendentes,percentual_estruturado
tb_credits_and_tags,106320,104949,1371,98.71
tb_movies_financials,106165,106165,0,100.0
tb_movies_info,106930,106585,345,99.68
tb_movies_metrics,107364,106456,908,99.15
tb_movies_reviews,32412,32412,0,100.0


Registros preservados somente nas auxiliares: 2624


#### Resultado da preparação dos arquivos

Os arquivos financeiros e de avaliações tiveram 100% dos registros encaminhados às tabelas principais, com 106.165 e 32.412 registros, respectivamente. A tabela de informações recebeu 106.585 registros, correspondentes a 99,68% da origem; métricas recebeu 106.456, ou 99,15%; e créditos recebeu 104.949, ou 98,71%. Permaneceram somente nas auxiliares 345 registros de informações, 908 de métricas e 1.371 de créditos, totalizando 2.624. Em todas as fontes, os registros aceitos somados aos pendentes correspondem ao total recebido. 

Esses percentuais medem a compatibilidade estrutural com a política adotada, não a qualidade dos valores. A cobertura das tabelas principais permanece parcial, e os registros pendentes continuam preservados para revisão.

## Consulta à API do Banco Central

Vamos consultar as cotações do dólar no período definido pelos widgets. A consulta utiliza limite de tempo, novas tentativas para falhas temporárias e paginação. Uma resposta inválida ou sem cotações interromperá a execução antes da gravação das cargas. Os dados recebidos serão utilizados na Silver para preparar a série diária de cotações e converter os valores financeiros dos filmes para reais.

In [0]:
url_api = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo("
    "dataInicial=@dataInicial,"
    "dataFinalCotacao=@dataFinalCotacao)"
)

parametros_api = {
    "@dataInicial": f"'{data_inicio.strftime('%m-%d-%Y')}'",
    "@dataFinalCotacao": f"'{data_fim.strftime('%m-%d-%Y')}'",
    "$select": "dataHoraCotacao,cotacaoCompra",
    "$format": "json",
    "$orderby": "dataHoraCotacao",
    "$top": 1000,
    "$skip": 0,
}

tentativas = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)

cotacoes = []
paginas_recebidas = set()

with requests.Session() as sessao:
    sessao.mount(
        "https://",
        HTTPAdapter(max_retries=tentativas),
    )

    while True:
        resposta = sessao.get(
            url_api,
            params=parametros_api,
            timeout=(10, 60),
        )

        resposta.raise_for_status()

        conteudo = json.loads(
            resposta.text,
            parse_float=Decimal,
        )

        if (
            not isinstance(conteudo, dict)
            or not isinstance(conteudo.get("value"), list)
        ):
            raise ValueError(
                "Estrutura inesperada na resposta da API."
            )

        pagina = conteudo["value"]

        if not pagina:
            break

        # Evita repetir uma página indefinidamente.
        assinatura_pagina = hashlib.sha256(
            json.dumps(
                pagina,
                sort_keys=True,
                default=str,
            ).encode("utf-8")
        ).hexdigest()

        if assinatura_pagina in paginas_recebidas:
            raise ValueError(
                "A API repetiu uma página. Confira a paginação."
            )

        paginas_recebidas.add(assinatura_pagina)
        cotacoes.extend(pagina)

        parametros_api["$skip"] += len(pagina)

if not cotacoes:
    raise ValueError(
        "Nenhuma cotação encontrada. Revise o período dos widgets, "
        "incluindo dias úteis anteriores."
    )

print("Período consultado:", data_inicio, "até", data_fim)
print("Registros recebidos:", len(cotacoes))

Período consultado: 2026-09-15 até 2026-09-21
Registros recebidos: 4


#### Resultado da consulta ao Banco Central

Na execução registrada, a consulta de 13/09/2026 a 19/09/2026 retornou cinco cotações. A quantidade recebida corresponde ao período consultado e pode ser menor que o total acumulado na tabela, que também preserva consultas anteriores.

## Preparação das cotações

Vamos conferir os campos recebidos, verificar se suas datas pertencem ao período solicitado e garantir que os valores possam ser representados como DECIMAL(18,8) sem arredondamento. Manteremos dataHoraCotacao como texto e cotacaoCompra como decimal. O preenchimento dos dias sem cotação ficará para a Silver.

In [0]:
registros_cotacao = []

for registro in cotacoes:
    if (
        not isinstance(registro, dict)
        or set(registro) != {
            "dataHoraCotacao",
            "cotacaoCompra",
        }
    ):
        raise ValueError("A API retornou campos inesperados.")

    data_hora = registro["dataHoraCotacao"]
    valor = registro["cotacaoCompra"]

    if not isinstance(data_hora, str) or not data_hora:
        raise ValueError(
            "dataHoraCotacao ausente ou inválida."
        )

    # Validamos a data sem substituir o texto recebido.
    data_registro = datetime.fromisoformat(data_hora).date()

    if not data_inicio <= data_registro <= data_fim:
        raise ValueError(
            "A API retornou uma cotação fora do período."
        )

    if (
        isinstance(valor, bool)
        or not isinstance(valor, (Decimal, int))
    ):
        raise ValueError(
            "cotacaoCompra ausente ou não numérica."
        )

    valor = Decimal(valor)

    if (
        not valor.is_finite()
        or abs(valor) >= Decimal("10000000000")
    ):
        raise ValueError(
            "Cotação incompatível com DECIMAL(18,8)."
        )

    if valor != valor.quantize(Decimal("0.00000001")):
        raise ValueError(
            "A cotação exige mais de oito casas decimais. "
            "Revise o tipo antes de gravar."
        )

    registros_cotacao.append((data_hora, valor))

estrutura_cotacao = StructType([
    StructField("dataHoraCotacao", StringType(), True),
    StructField("cotacaoCompra", DecimalType(18, 8), True),
])

df_cotacao = spark.createDataFrame(
    registros_cotacao,
    estrutura_cotacao,
)

display(
    df_cotacao.orderBy("dataHoraCotacao")
)

# A ordenação serve apenas para identificar o conteúdo recebido.
conteudo_cotacao = json.dumps(
    {
        "data_inicio": data_inicio.isoformat(),
        "data_fim": data_fim.isoformat(),
        "registros": sorted(
            (data, str(valor))
            for data, valor in registros_cotacao
        ),
    },
    sort_keys=True,
)

# Evita repetir a entrada na lista ao reexecutar esta célula.
cargas = [
    carga
    for carga in cargas
    if carga["tabela"] != "tb_cotacao_dolar"
]

cargas.append({
    "tabela": "tb_cotacao_dolar",
    "df": df_cotacao,
    "quantidade": len(registros_cotacao),
    "assinatura": hashlib.sha256(
        conteudo_cotacao.encode("utf-8")
    ).hexdigest(),
    "origem": (
        f"{url_api} | período: "
        f"{data_inicio.isoformat()} a {data_fim.isoformat()}"
    ),
    "politica": "ptax_periodo_v1",
    "total_origem": len(registros_cotacao),
    "total_pendente": 0,
})

dataHoraCotacao,cotacaoCompra
2026-09-15 13:09:19.199664,5.14840000
2026-09-16 13:05:30.35873,5.15200000
2026-09-17 13:03:21.858212,5.15150000
2026-09-18 13:03:34.742036,5.15690000


#### Resultado da preparação das cotações

As cinco cotações recebidas, referentes aos dias 14 a 18 de setembro de 2026, passaram pelas verificações de estrutura, período e representação decimal. Os registros foram preparados para gravação sem preencher as datas sem retorno da API. Esses resultados descrevem a execução salva e poderão mudar em consultas de outros períodos.

## Verificação dos destinos

Antes de gravar, vamos conferir se as onze cargas esperadas foram preparadas: cinco tabelas principais dos CSVs, cinco auxiliares e uma tabela de cotações. Também verificaremos novamente as assinaturas dos arquivos e o formato e a estrutura das tabelas existentes. O schema Bronze será criado caso ainda não exista. Não faremos alterações automáticas de estrutura. Os arquivos do volume deverão permanecer inalterados durante toda a execução, pois o Spark pode relê-los ao executar as operações.

In [0]:
tabelas_esperadas = {"tb_cotacao_dolar"}

for fonte in fontes:
    tabelas_esperadas.add(fonte["tabela"])
    tabelas_esperadas.add(f"{fonte['tabela']}_raw")

nomes_preparados = [carga["tabela"] for carga in cargas]

if (
    len(nomes_preparados) != len(set(nomes_preparados))
    or set(nomes_preparados) != tabelas_esperadas
):
    raise ValueError(
        "As cargas preparadas não correspondem às onze tabelas "
        "esperadas. Reexecute a preparação dos CSVs e da API."
    )

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_bronze}"
)

arquivos_conferidos = set()
resumo_destinos = []

for carga in cargas:
    origem = carga["origem"]

    if (
        origem.startswith("/Volumes/")
        and origem not in arquivos_conferidos
    ):
        if calcular_hash_arquivo(origem) != carga["assinatura"]:
            raise ValueError(
                f"O arquivo mudou durante a execução: {origem}. "
                "Reexecute a preparação das entradas."
            )

        arquivos_conferidos.add(origem)

    # Guardamos a estrutura para reutilizá-la na conferência final.
    tipos_entrada = carga["df"].dtypes

    carga["colunas"] = [
        coluna for coluna, _ in tipos_entrada
    ]

    carga["estrutura_esperada"] = (
        tipos_entrada
        + [("ingestion_datetime", "timestamp")]
    )

    destino = f"{catalogo}.{schema_bronze}.{carga['tabela']}"
    existe = spark.catalog.tableExists(destino)

    if existe:
        detalhe = spark.sql(
            f"DESCRIBE DETAIL {destino}"
        ).first()

        if detalhe["format"].lower() != "delta":
            raise ValueError(
                f"O destino não está em Delta: {destino}"
            )

        if spark.table(destino).dtypes != carga["estrutura_esperada"]:
            raise ValueError(
                f"Estrutura incompatível: {destino}"
            )

    resumo_destinos.append((
        destino,
        "Existente e compatível" if existe else "Será criado",
    ))

display(
    spark.createDataFrame(
        resumo_destinos,
        ["tabela", "situacao_destino"],
    )
)

print("As onze cargas passaram pela verificação dos destinos.")

tabela,situacao_destino
workspace.bronze.tb_movies_info_raw,Existente e compatível
workspace.bronze.tb_movies_info,Existente e compatível
workspace.bronze.tb_movies_financials_raw,Existente e compatível
workspace.bronze.tb_movies_financials,Existente e compatível
workspace.bronze.tb_movies_metrics_raw,Existente e compatível
workspace.bronze.tb_movies_metrics,Existente e compatível
workspace.bronze.tb_credits_and_tags_raw,Existente e compatível
workspace.bronze.tb_credits_and_tags,Existente e compatível
workspace.bronze.tb_movies_reviews_raw,Existente e compatível
workspace.bronze.tb_movies_reviews,Existente e compatível


As onze cargas passaram pela verificação dos destinos.


#### Resultado da verificação dos destinos

As onze cargas esperadas foram identificadas, e as assinaturas dos arquivos permaneceram iguais às calculadas na preparação. As tabelas existentes passaram pelas verificações de formato Delta e compatibilidade de estrutura. Nenhuma incompatibilidade foi encontrada nessa etapa, permitindo continuar para a comparação dos dados e a gravação.

## Gravação e reexecução

Vamos adicionar ingestion_datetime no momento da gravação e utilizar o modo append. Para os CSVs fixos desta atividade, compararemos o conteúdo preparado com o armazenado, incluindo a quantidade de repetições e desconsiderando apenas o horário de ingestão. Conteúdos iguais não serão inseridos novamente; diferenças em tabelas já preenchidas interromperão a execução para revisão. 

Nas cotações, acrescentaremos somente combinações de data, hora e valor ainda não armazenadas. Cada tabela será gravada separadamente, com identificação de transação e metadados da carga. Executaremos uma carga por vez. Essa estratégia permite repetir a mesma entrada, mas não implementa a atualização automática dos CSVs quando seu conteúdo muda.

In [0]:
import hashlib
import json

from pyspark.sql import functions as F

plano_gravacao = []

# Conferimos todos os destinos antes de iniciar as gravações.
for carga in cargas:
    destino = f"{catalogo}.{schema_bronze}.{carga['tabela']}"
    df_carga = carga["df"]
    colunas = df_carga.columns

    existe = spark.catalog.tableExists(destino)

    if existe:
        df_existente = spark.table(destino).select(*colunas)
        total_anterior = df_existente.count()
    else:
        df_existente = df_carga.limit(0)
        total_anterior = 0

    if carga["tabela"] == "tb_cotacao_dolar":
        # Uma consulta pode trazer dias já consultados anteriormente.
        chaves = ["dataHoraCotacao", "cotacaoCompra"]

        df_novos = df_carga.dropDuplicates(chaves)

        if existe:
            df_novos = df_novos.join(
                df_existente.select(*chaves).distinct(),
                on=chaves,
                how="left_anti",
            )

        df_novos = df_novos.select(*colunas)
        quantidade_nova = df_novos.count()

    elif total_anterior == 0:
        df_novos = df_carga
        quantidade_nova = carga["quantidade"]

    else:
        # exceptAll considera também quantas vezes cada linha aparece.
        # Não removemos duplicidades que já vieram do CSV.
        possui_diferenca = (
            df_carga.exceptAll(df_existente).limit(1).count() > 0
            or df_existente.exceptAll(df_carga).limit(1).count() > 0
        )

        if possui_diferenca:
            raise ValueError(
                f"{destino}: o conteúdo preparado é diferente "
                "do conteúdo armazenado. Nenhuma gravação desta "
                "execução foi iniciada. Revise a origem e a leitura."
            )

        df_novos = df_carga.limit(0)
        quantidade_nova = 0

    plano_gravacao.append({
        "carga": carga,
        "destino": destino,
        "df_novos": df_novos,
        "quantidade_nova": quantidade_nova,
        "total_anterior": total_anterior,
    })


resultados_gravacao = []

for item in plano_gravacao:
    carga = item["carga"]
    destino = item["destino"]
    quantidade_nova = item["quantidade_nova"]

    if quantidade_nova > 0:
        # A identificação deixa de depender da data ou do id_lote.
        identidade = json.dumps(
            {
                "destino": destino,
                "assinatura": carga["assinatura"],
                "politica": carga["politica"],
            },
            sort_keys=True,
        )

        identificador = hashlib.sha256(
            identidade.encode("utf-8")
        ).hexdigest()

        metadados = json.dumps(
            {
                "id_lote": id_lote,
                "origem": carga["origem"],
                "sha256": carga["assinatura"],
                "politica_leitura": carga["politica"],
                "registros_recebidos": carga["quantidade"],
                "registros_previstos_para_gravacao": quantidade_nova,
            },
            ensure_ascii=False,
        )

        (
            item["df_novos"]
            .withColumn("ingestion_datetime", F.current_timestamp())
            .write
            .format("delta")
            .mode("append")
            .option("txnAppId", f"cinedata:conteudo_v2:{identificador}")
            .option("txnVersion", 0)
            .option("userMetadata", metadados)
            .saveAsTable(destino)
        )

    total_final = spark.table(destino).count()
    adicionados = total_final - item["total_anterior"]

    if adicionados != quantidade_nova:
        raise ValueError(
            f"{destino}: esperávamos adicionar {quantidade_nova} "
            f"registros, mas a diferença foi {adicionados}. "
            "Verifique o histórico antes de continuar."
        )

    resultados_gravacao.append((
        destino,
        carga["quantidade"],
        adicionados,
        total_final,
    ))

display(
    spark.createDataFrame(
        resultados_gravacao,
        [
            "tabela",
            "registros_da_carga",
            "registros_adicionados",
            "total_acumulado",
        ],
    )
)

tabela,registros_da_carga,registros_adicionados,total_acumulado
workspace.bronze.tb_movies_info_raw,106930,0,106930
workspace.bronze.tb_movies_info,106585,0,106585
workspace.bronze.tb_movies_financials_raw,106165,0,106165
workspace.bronze.tb_movies_financials,106165,0,106165
workspace.bronze.tb_movies_metrics_raw,107364,0,107364
workspace.bronze.tb_movies_metrics,106456,0,106456
workspace.bronze.tb_credits_and_tags_raw,106320,0,106320
workspace.bronze.tb_credits_and_tags,104949,0,104949
workspace.bronze.tb_movies_reviews_raw,32412,0,32412
workspace.bronze.tb_movies_reviews,32412,0,32412


#### Resultado da gravação e da reexecução

Na execução registrada antes desta revisão, nenhuma das onze tabelas recebeu registros adicionais, pois o conteúdo preparado já estava armazenado. A consulta trouxe quatro cotações, enquanto a tabela manteve sete registros acumulados de consultas anteriores. Essa diferença não representa duplicação. O resultado confirmou o controle de repetição para aquela carga; uma nova consulta poderá acrescentar cotações ainda não armazenadas.

## Conferência das tabelas gravadas

Vamos verificar se todas as tabelas existem, estão em Delta, possuem a estrutura esperada e apresentam ingestion_datetime preenchido. Para os CSVs fixos desta atividade, o conteúdo armazenado deve corresponder ao preparado pela leitura atual. 

Para as cotações, o total acumulado pode superar o recebido nesta execução, pois a tabela também mantém registros de períodos consultados anteriormente. Essas verificações conferem a gravação; a cobertura das fontes será apresentada em seguida.

In [0]:
# Verificamos novamente os arquivos após a gravação.
for caminho in arquivos_conferidos:
    assinatura_esperada = next(
        carga["assinatura"]
        for carga in cargas
        if carga["origem"] == caminho
    )

    if calcular_hash_arquivo(caminho) != assinatura_esperada:
        raise ValueError(
            f"O arquivo mudou durante a execução: {caminho}. "
            "Revise a carga antes de continuar."
        )

# Esta lista também será utilizada na consulta ao histórico.
tabelas_para_conferir = [
    (carga["tabela"], carga["df"])
    for carga in cargas
]

conferencia_final = []
falhas_conferencia = []

for carga in cargas:
    nome_tabela = carga["tabela"]
    df_entrada = carga["df"]
    colunas = carga["colunas"]

    destino = f"{catalogo}.{schema_bronze}.{nome_tabela}"

    if not spark.catalog.tableExists(destino):
        raise ValueError(
            f"Tabela não encontrada: {destino}"
        )

    tabela = spark.table(destino)

    detalhe = spark.sql(
        f"DESCRIBE DETAIL {destino}"
    ).first()

    if detalhe["format"].lower() != "delta":
        raise ValueError(
            f"Formato inesperado: {destino}"
        )

    if tabela.dtypes != carga["estrutura_esperada"]:
        raise ValueError(
            f"Estrutura inesperada: {destino}"
        )

    resultado = tabela.agg(
        F.count("*").alias("total"),
        F.count(
            F.when(F.col("ingestion_datetime").isNull(), 1)
        ).alias("horarios_nulos"),
        F.max("ingestion_datetime").alias("ultima_ingestao"),
    ).first()

    df_salvo = tabela.select(*colunas)

    if nome_tabela == "tb_cotacao_dolar":
        chaves = ["dataHoraCotacao", "cotacaoCompra"]

        cotacoes_faltantes = (
            df_entrada.select(*chaves)
            .distinct()
            .join(
                df_salvo.select(*chaves).distinct(),
                on=chaves,
                how="left_anti",
            )
            .limit(1)
            .count()
        )

        cotacoes_repetidas = (
            df_salvo
            .groupBy(*chaves)
            .count()
            .filter(F.col("count") > 1)
            .limit(1)
            .count()
        )

        conteudo_confere = (
            cotacoes_faltantes == 0
            and cotacoes_repetidas == 0
        )

        criterio = "Carga presente e sem cotações repetidas"

    else:
        registros_faltantes = (
            df_entrada.exceptAll(df_salvo)
            .limit(1)
            .count()
        )

        registros_extras = (
            df_salvo.exceptAll(df_entrada)
            .limit(1)
            .count()
        )

        conteudo_confere = (
            registros_faltantes == 0
            and registros_extras == 0
        )

        criterio = "Conteúdo e repetições iguais à entrada"

    if (
        resultado["horarios_nulos"] > 0
        or not conteudo_confere
    ):
        falhas_conferencia.append(destino)

    conferencia_final.append((
        destino,
        detalhe["format"],
        resultado["total"],
        resultado["horarios_nulos"],
        conteudo_confere,
        criterio,
        resultado["ultima_ingestao"],
    ))

estrutura_resumo = (
    "tabela STRING, "
    "formato STRING, "
    "total_registros LONG, "
    "horarios_nulos LONG, "
    "conteudo_confere BOOLEAN, "
    "criterio_conferencia STRING, "
    "ultima_ingestao_utc TIMESTAMP"
)

display(
    spark.createDataFrame(
        conferencia_final,
        schema=estrutura_resumo,
    )
)

if falhas_conferencia:
    raise ValueError(
        "Falha na conferência final: "
        + ", ".join(falhas_conferencia)
    )

print(
    "As onze tabelas passaram pela conferência "
    "de estrutura, conteúdo e horário de ingestão."
)

tabela,formato,total_registros,horarios_nulos,conteudo_confere,criterio_conferencia,ultima_ingestao_utc
workspace.bronze.tb_movies_info_raw,delta,106930,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:53:22.304Z
workspace.bronze.tb_movies_info,delta,106585,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T22:44:31.075Z
workspace.bronze.tb_movies_financials_raw,delta,106165,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:53:44.129Z
workspace.bronze.tb_movies_financials,delta,106165,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:53:48.943Z
workspace.bronze.tb_movies_metrics_raw,delta,107364,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:28:07.947Z
workspace.bronze.tb_movies_metrics,delta,106456,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T22:44:51.644Z
workspace.bronze.tb_credits_and_tags_raw,delta,106320,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:53:55.869Z
workspace.bronze.tb_credits_and_tags,delta,104949,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T22:45:01.357Z
workspace.bronze.tb_movies_reviews_raw,delta,32412,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:54:08.359Z
workspace.bronze.tb_movies_reviews,delta,32412,0,true,Conteúdo e repetições iguais à entrada,2026-09-19T19:54:12.131Z


As onze tabelas passaram pela conferência de estrutura, conteúdo e horário de ingestão.


#### Resultado da conferência final

As onze tabelas passaram pelas verificações de estrutura, conteúdo e preenchimento de ingestion_datetime. Nas tabelas dos CSVs, o conteúdo armazenado corresponde ao preparado, incluindo a quantidade de repetições de cada linha. Na tabela de cotações, todas as combinações recebidas estão presentes, sem repetição de data, hora e valor, preservando os registros de consultas anteriores. 

As assinaturas dos arquivos também permaneceram iguais após a gravação. Essa conferência confirma a persistência dos dados preparados, mas não elimina as pendências de interpretação nem comprova a qualidade dos valores.

## Histórico das operações

Vamos consultar a última operação registrada em cada tabela, sua versão e seu horário. O histórico pode mostrar uma gravação, uma restauração ou uma otimização dos arquivos. 

Os metadados personalizados descrevem a carga quando são enviados pela operação de gravação; sua ausência em uma operação de manutenção não indica falta de dados. Quando a reexecução não acrescenta registros, não ocorre uma nova gravação e o histórico continua mostrando a operação anterior.

In [0]:
historico_resumido = []

for nome_tabela, _ in tabelas_para_conferir:
    destino = (
        f"{catalogo}.{schema_bronze}.{nome_tabela}"
    )

    ultima_operacao = (
        spark.sql(f"DESCRIBE HISTORY {destino}")
        .orderBy(F.col("version").desc())
        .first()
    )

    historico_resumido.append((
        destino,
        ultima_operacao["version"],
        ultima_operacao["timestamp"],
        ultima_operacao["operation"],
        ultima_operacao["userMetadata"],
    ))

estrutura_historico = (
    "tabela STRING, "
    "versao LONG, "
    "horario TIMESTAMP, "
    "operacao STRING, "
    "metadados_carga STRING"
)

df_historico = spark.createDataFrame(
    historico_resumido,
    schema=estrutura_historico,
)

display(df_historico)

sem_metadados = sum(
    linha[4] is None or linha[4] == ""
    for linha in historico_resumido
)

if sem_metadados:
    print(
        "Tabelas cuja última operação não contém "
        "metadados personalizados:",
        sem_metadados,
    )

tabela,versao,horario,operacao,metadados_carga
workspace.bronze.tb_movies_info_raw,2,2026-09-19T22:02:27.000Z,RESTORE,null
workspace.bronze.tb_movies_info,3,2026-09-19T22:44:36.000Z,WRITE,"{""motivo"": ""Ampliação da leitura estrutural"", ""politica"": ""leitura_prioritaria_sem_escape_v2"", ""sha256"": ""99eb606a38174051f3e647483cc3bf96bb431385d15c5a200cdc9ef95e4a5e71"", ""registros_previstos"": 5275}"
workspace.bronze.tb_movies_financials_raw,2,2026-09-19T22:02:44.000Z,RESTORE,null
workspace.bronze.tb_movies_financials,2,2026-09-19T22:02:52.000Z,RESTORE,null
workspace.bronze.tb_movies_metrics_raw,2,2026-09-19T22:03:00.000Z,RESTORE,null
workspace.bronze.tb_movies_metrics,3,2026-09-19T22:44:55.000Z,WRITE,"{""motivo"": ""Ampliação da leitura estrutural"", ""politica"": ""leitura_prioritaria_sem_escape_v2"", ""sha256"": ""ad20d13ea8d363a6c2fd6a9e8b3e7082fc729f59e78a588e99e80ca53bfe12cd"", ""registros_previstos"": 1375}"
workspace.bronze.tb_credits_and_tags_raw,2,2026-09-19T22:03:14.000Z,RESTORE,null
workspace.bronze.tb_credits_and_tags,3,2026-09-19T22:45:05.000Z,WRITE,"{""motivo"": ""Ampliação da leitura estrutural"", ""politica"": ""leitura_prioritaria_sem_escape_v2"", ""sha256"": ""63f0225044d27ae3dc148de45aa88072b2e69f578ad28322dc12c09a6f1c16f8"", ""registros_previstos"": 736}"
workspace.bronze.tb_movies_reviews_raw,2,2026-09-19T22:03:27.000Z,RESTORE,null
workspace.bronze.tb_movies_reviews,2,2026-09-19T22:03:35.000Z,RESTORE,null


Tabelas cuja última operação não contém metadados personalizados: 8


#### Resultado da consulta ao histórico

Na execução registrada, o histórico apresentou gravações e operações de manutenção realizadas durante o desenvolvimento. Oito tabelas não exibiram metadados personalizados na última operação. Essa ausência deve ser interpretada junto ao tipo da operação e não significa que seus dados estejam ausentes. Novas inserções poderão alterar o histórico apresentado.

## Conclusão e reexecução da Bronze

A Bronze reúne cinco tabelas principais dos CSVs, cinco auxiliares de preservação e uma tabela de cotações. A rotina utiliza append e evita inserir novamente o conteúdo já armazenado, preservando as repetições presentes nos próprios arquivos. Para reexecutar, devemos rodar o notebook desde o início, manter os CSVs inalterados durante o processamento e executar uma carga por vez. 

Alterações nos arquivos ou na política de leitura exigem revisão antes de uma nova carga. Permanecem 2.624 registros apenas nas auxiliares: 345 de informações, 908 de métricas e 1.371 de créditos. Essa limitação impede considerar a ingestão integralmente atendida e afeta a cobertura das próximas camadas. A compatibilidade estrutural também não garante que cada valor esteja na coluna correta. Os tratamentos de negócio ficam na Silver, e a organização para análise e contexto de IA fica na Gold.